# AI Company Brochure Generator

Build a company brochure from its public website using locally hosted language models.

**Author:** Manish Kundu

This project uses Python, web-page extraction, and Ollama to identify relevant company pages and generate a concise Markdown brochure.

> Learning project adapted from an AI/LLM engineering course exercise. The implementation has been adapted to use local Ollama models.


In [ ]:
# Imports

import json
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI  # Ollama exposes an OpenAI-compatible local API


In [ ]:
# Configure the local Ollama client

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "deepseek-r1:1.5b"  # Change to "llama3.2" if preferred

client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"  # Placeholder required by the client; Ollama does not validate it
)

print(f"Using local Ollama model: {MODEL}")

# Set the company you want to create a brochure for
COMPANY_NAME = "Hugging Face"
COMPANY_URL = "https://huggingface.co"


In [ ]:
links = fetch_website_links(COMPANY_URL)
links

## Step 1: Identify relevant website links

Ask the language model to review links found on the company's website and return useful pages, such as About, Products, and Careers pages, as structured JSON.


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt(COMPANY_URL))

In [ ]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model="deepseek-r1:1.5b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
# Optional: Check that Ollama is available

import requests

response = requests.get("http://localhost:11434/api/tags", timeout=10)

if response.ok:
    print("Ollama is running!")
    print("Available models:", [model["name"] for model in response.json().get("models", [])])
else:
    print("Could not connect to Ollama. Start Ollama and try again.")


In [ ]:
select_relevant_links(COMPANY_URL)

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} using {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
select_relevant_links(COMPANY_URL)

In [ ]:
select_relevant_links(COMPANY_URL)

## Step 2: Generate the company brochure

Combine the landing page and selected pages, then ask the local model to produce a concise brochure in Markdown.


In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links(COMPANY_URL))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt(COMPANY_NAME, COMPANY_URL)

In [ ]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))


In [ ]:
create_brochure(COMPANY_NAME, COMPANY_URL)

## Step 3: Stream the generated brochure

Stream the model's response so the brochure appears progressively in the notebook.


In [ ]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
stream_brochure(COMPANY_NAME, COMPANY_URL)

In [ ]:
stream_brochure(COMPANY_NAME, COMPANY_URL)

## Project notes

This notebook demonstrates a simple multi-step LLM workflow:

1. Extract links from a company's website.
2. Use a local language model to select relevant pages.
3. Collect page content.
4. Generate a company brochure in Markdown.
5. Optionally stream the generated response.

The model runs locally through Ollama. Website content is still retrieved from the public internet.
